# Spatial Cellular Graph Learning: Complete Pipeline

Run the sections from top to bottom. Each stage calls one numbered implementation script from the same directory.

Stages:
1. Initialize the project
2. Prepare HE-enhanced region tables
3. Build spatial graphs
4. Train the multi-task GIN
5. Extract embeddings and cluster MicroE states
6. Run B-T permutation analysis
7. Calculate B-T edge density


## 1. Initialize the project

The scripts use project-relative directories defined in `config.py`. Place input data under the configured `data/` directory before continuing.


In [ ]:
from pathlib import Path
import runpy
import sys

pipeline_candidates = [Path.cwd() / 'PCF', Path.cwd()]
PIPELINE_DIR = next(
    path for path in pipeline_candidates
    if (path / 'config.py').exists() and (path / 'common.py').exists()
)
sys.path.insert(0, str(PIPELINE_DIR.resolve()))
print(f'Pipeline directory: {PIPELINE_DIR}')


## 2. Prepare HE-enhanced region tables

This stage matches QuPath HE measurements to the existing cell tables, synchronizes `CELL_ID` values across modalities, and writes processed tables without overwriting the originals.


In [ ]:
runpy.run_path(str(PIPELINE_DIR / '01_preprocess_he.py'), run_name='__main__')


## 3. Build spatial graphs

This stage creates one serialized spatial graph for every labeled region. Each graph contains cell nodes, node attributes, spatial edges, and edge attributes.


In [ ]:
runpy.run_path(str(PIPELINE_DIR / '02_build_graphs.py'), run_name='__main__')


## 4. Train the multi-task GIN

The model uses three-hop cellular subgraphs and has two supervised tasks:

- Predict the masked cell type of the center cell.
- Predict the region-level `Transfer_type`.

The training script creates stratified train, validation, and test region splits and saves the checkpoint under `outputs/models/`. This stage can take substantially longer than the other stages.


In [ ]:
runpy.run_path(str(PIPELINE_DIR / '03_train_gnn.py'), run_name='__main__')


## 5. Extract embeddings and define MicroE states

This stage samples local subgraphs, extracts graph embeddings from the trained model, applies PCA and KMeans, and writes MicroE assignments and embedding arrays.


In [ ]:
runpy.run_path(str(PIPELINE_DIR / '04_analyze_embeddings.py'), run_name='__main__')


## 6. Run B-T spatial permutation analysis

This optional stage tests whether the model prediction changes after redistributing B-cell and T-cell attributes while preserving spatial coordinates. Set `PERMUTATION_REGION_ID` and `PERMUTATION_TARGET_LABEL` in `config.py` before running this section.


In [ ]:
runpy.run_path(str(PIPELINE_DIR / '05_bt_permutation.py'), run_name='__main__')


## 7. Calculate B-T edge density

This optional stage counts B-T edges and B/T cell numbers for every region and writes a region-level summary table.


In [ ]:
runpy.run_path(str(PIPELINE_DIR / '06_bt_edge_analysis.py'), run_name='__main__')


## Outputs

The main outputs are:

- `data/processed_regions/`: synchronized cell tables
- `outputs/example_dataset/graph/`: serialized region graphs
- `outputs/region_splits.npz`: region-level train/validation/test split
- `outputs/models/gin_he/model_final.pt`: trained model checkpoint
- `outputs/analysis/embeddings.npz`: graph embeddings and cluster results
- `outputs/analysis/microenvironment_assignments.csv`: MicroE assignments
- `outputs/analysis/bt_permutation_summary.json`: permutation summary
- `outputs/analysis/bt_edge_counts.csv`: B-T edge statistics
